# 実験: Exp-BFly-A-v2（Fly_Level + 全バタフライ水準追加）

**目的**: B2〜B7 全ての水準を全行に追加し、かつ Fly_Level（自分自身の水準）も残した場合の効果を確認する。

In [ ]:
import sys
sys.path.insert(0, '..')
import warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from src.processing        import load_and_clean_data
from src.features_rv       import generate_rv_features
from src.pooling_butterfly import pool_butterfly_data
from src.modeling          import walk_forward_with_model, summarize_ic

START_DATE = '2024-01-01'
INSTRUMENT_INDICES = set(range(2, 8))

print('Loading data...')
df_raw = load_and_clean_data('../data/BOJ_data.xlsx', '../data/BOJ_meeting_history.csv')
df_rv  = generate_rv_features(df_raw)
df_fly_baseline = pool_butterfly_data(df_rv)

def pool_butterfly_data_exp_a_v2(feat_df: pd.DataFrame) -> pd.DataFrame:
    df = feat_df.copy()
    b_level_cols = []
    for n in range(2, 8):
        col_name = f'B{n}_level'
        df[col_name] = df[f'B{n}']
        b_level_cols.append(col_name)
    fly_raw_cols = [f'B{n}' for n in range(2, 8)]
    id_cols = [c for c in df.columns if c not in fly_raw_cols]
    pooled = df.melt(id_vars=id_cols, value_vars=fly_raw_cols, var_name='Rate_Label', value_name='Rate_Value')
    pooled['Meeting_Index'] = pooled['Rate_Label'].str.extract('(\d+)').astype(int)
    pooled = pooled.sort_values(['Rate_Label', 'Date']).reset_index(drop=True)
    for h in [1, 3, 5]:
        pooled[f'Target_{h}d'] = pooled.groupby('Rate_Label')['Rate_Value'].shift(-h) - pooled['Rate_Value']
    for h in [3, 5]:
        instr_std = pooled.groupby('Rate_Label')[f'Target_{h}d'].transform('std')
        pooled[f'Target_{h}d_std']  = instr_std
        pooled[f'Target_{h}d_norm'] = pooled[f'Target_{h}d'] / instr_std
    
    pooled['Fly_Level'] = pooled['Rate_Value']
    
    imputed_cols_fly = []
    for n in range(2, 8):
        for col in [f'M{n-1}_is_imputed', f'M{n}_is_imputed', f'M{n+1}_is_imputed']:
            if col in pooled.columns and col not in imputed_cols_fly: imputed_cols_fly.append(col)

    basic_cols   = ['Meeting_Index', 'is_post_mpm', 'Days_to_MPM', 'Actual_Policy_Rate']
    anchor_cols  = ['M1_spread', 'M1_frac_diff', 'Slope_M1M8', 'Slope_M1M8_frac_diff']
    fly_fd_cols  = [f'B{n}_frac_diff' for n in range(2, 8)]
    ext_fd_cols  = ['USDJPY_frac_diff', 'JGB_Future_frac_diff', 'Nikkei225_frac_diff', 'DXY_frac_diff']
    target_cols = ['Date', 'Target_1d_norm', 'Target_3d_norm', 'Target_5d_norm', 'Target_1d_std', 'Target_3d_std', 'Target_5d_std']
    
    final_cols = basic_cols + ['Fly_Level'] + b_level_cols + anchor_cols + fly_fd_cols + imputed_cols_fly + ext_fd_cols + target_cols
    available = [c for c in final_cols if c in pooled.columns]
    return pooled[available]

df_fly_exp_v2 = pool_butterfly_data_exp_a_v2(df_rv)

print('Running walk-forward...')
res_3d_b, _, _, _, _ = walk_forward_with_model(df_fly_baseline, 'Target_3d_norm', START_DATE)
ic3_b = summarize_ic(res_3d_b, instrument_indices=INSTRUMENT_INDICES)
res_3d_v2, mdl_3d_v2, _, _, _ = walk_forward_with_model(df_fly_exp_v2, 'Target_3d_norm', START_DATE)
ic3_v2 = summarize_ic(res_3d_v2, instrument_indices=INSTRUMENT_INDICES)

print(f'\nBaseline 3d Global IC: {ic3_b["ic_all"]:.4f}')
print(f'Exp-A-v2 3d Global IC: {ic3_v2["ic_all"]:.4f}')

imp = pd.DataFrame({'feature': mdl_3d_v2.feature_name(),
                    'gain':    mdl_3d_v2.feature_importance(importance_type='gain')})
print("\nTop 20 Features (Exp-A-v2):")
print(imp.sort_values('gain', ascending=False).head(20).to_string(index=False))
